In [2]:
# classical_pd.py
import itertools, json, statistics

# C = 0 (cooperate), D = 1 (defect)
PAYOFF = {
    (0,0): (3,3), (0,1): (0,5),
    (1,0): (5,0), (1,1): (1,1)
}

def is_nash(a, b):
    """Check if (a,b) is a mutual best response."""
    # Player 1: would deviating flip his payoff?
    p1_stays = PAYOFF[(a,b)][0] >= PAYOFF[(1-a,b)][0]
    # Player 2: would deviating flip her payoff?
    p2_stays = PAYOFF[(a,b)][1] >= PAYOFF[(a,1-b)][1]
    return p1_stays and p2_stays

def brute_force():
    # Build a JSON-friendly table
    table = []
    equilibria = []
    for a, b in itertools.product([0,1], repeat=2):
        payoff = PAYOFF[(a,b)]
        entry = {"actions": [a, b], "payoff": payoff}
        table.append(entry)
        if is_nash(a, b):
            equilibria.append(entry)

    # Basic stats on payoffs
    first_payoffs  = [e["payoff"][0] for e in table]
    second_payoffs = [e["payoff"][1] for e in table]
    stats = {
        "first_mean":  statistics.mean(first_payoffs),
        "first_stdev": statistics.pstdev(first_payoffs),
        "second_mean":  statistics.mean(second_payoffs),
        "second_stdev": statistics.pstdev(second_payoffs),
    }

    results = {
        "table": table,
        "nash_equilibria": equilibria,
        "stats": stats
    }

    with open("results_classical_pd.json", "w") as f:
        json.dump(results, f, indent=2)

    return results

if __name__ == "__main__":
    res = brute_force()
    print("✅ Wrote results_classical_pd.json")
    print("ℹ️ Nash equilibria:", res["nash_equilibria"])
    print("ℹ️ Payoff stats:", res["stats"])


✅ Wrote results_classical_pd.json
ℹ️ Nash equilibria: [{'actions': [1, 1], 'payoff': (1, 1)}]
ℹ️ Payoff stats: {'first_mean': 2.25, 'first_stdev': 1.920286436967152, 'second_mean': 2.25, 'second_stdev': 1.920286436967152}


We want to demonstrate:
Algorithmic scaling, not raw timing
exactness vs sampling noise
qubit count (n) mapes to state-space (2^n), versus classical enumeration's 2^k profiles.

In [10]:
# classical_pd2_timed.py
import itertools
import time
import json

# 2-player Prisoner's Dilemma payoff matrix
# C = 0 (cooperate), D = 1 (defect)
PAYOFF = {
    (0, 0): (3, 3),
    (0, 1): (0, 5),
    (1, 0): (5, 0),
    (1, 1): (1, 1)
}

def is_nash(a, b):
    """True if (a,b) is a Nash equilibrium."""
    # unilateral deviations
    alt_A = PAYOFF[(1 - a, b)][0]
    alt_B = PAYOFF[(a, 1 - b)][1]
    return (PAYOFF[(a, b)][0] >= alt_A) and (PAYOFF[(a, b)][1] >= alt_B)

def enumerate_nash():
    equilibria = []
    for a, b in itertools.product([0, 1], repeat=2):
        if is_nash(a, b):
            equilibria.append((a, b))
    return equilibria

if __name__ == "__main__":
    t0 = time.time()
    nash_eq = enumerate_nash()
    elapsed = time.time() - t0

    # Console output
    print("2-Player PD Nash equilibria:", nash_eq)
    print(f"Enumeration time: {elapsed:.6f} seconds")

    # JSON dump
    with open("pd2_results.json", "w") as f:
        json.dump(
            {
                "equilibria": nash_eq,
                "time_seconds": elapsed
            },
            f,
            indent=2
        )


2-Player PD Nash equilibria: [(1, 1)]
Enumeration time: 0.000037 seconds


In [4]:
# classical_pd3.py
import itertools
import time
import json

# 2-player PD payoffs (C=0, D=1): (payoff_P1, payoff_P2)
PAYOFF2 = {
    (0,0):(3,3), (0,1):(0,5),
    (1,0):(5,0), (1,1):(1,1)
}

def pd3_payoff(bits):
    """Sum pairwise PD payoffs for each of 3 players."""
    pay = [0,0,0]
    for i,j in [(0,1),(0,2),(1,2)]:
        pi, pj = PAYOFF2[(bits[i], bits[j])]
        pay[i] += pi
        pay[j] += pj
    return tuple(pay)

def is_nash(bits):
    """Check no single player can improve by flipping their bit."""
    current = pd3_payoff(bits)
    for idx in range(3):
        flipped = list(bits)
        flipped[idx] ^= 1
        if pd3_payoff(tuple(flipped))[idx] > current[idx]:
            return False
    return True

def find_nash():
    eq = []
    for bits in itertools.product([0,1], repeat=3):
        if is_nash(bits):
            eq.append(bits)
    return eq

if __name__ == "__main__":
    t0 = time.time()
    equilibria = find_nash()
    elapsed = time.time() - t0

    print("3-Player PD Nash equilibria:", equilibria)
    print(f"Classical enumeration time: {elapsed:.6f} seconds")

    with open("pd3_equilibria.json", "w") as f:
        json.dump({
            "equilibria": equilibria,
            "time_seconds": elapsed
        }, f, indent=2)


3-Player PD Nash equilibria: [(1, 1, 1)]
Classical enumeration time: 0.000104 seconds


In [13]:
# classical_pd4_timed.py
import itertools
import time
import json
from itertools import combinations

# 2-player PD payoffs (C=0, D=1)
PAYOFF2 = {
    (0,0):(3,3), (0,1):(0,5),
    (1,0):(5,0), (1,1):(1,1)
}

def pd4_payoff(bits):
    """Sum pairwise PD payoffs for each of 4 players."""
    pay = [0]*4
    for i, j in combinations(range(4), 2):
        pi, pj = PAYOFF2[(bits[i], bits[j])]
        pay[i] += pi
        pay[j] += pj
    return tuple(pay)

def is_nash(bits):
    """Check no single player can improve by unilaterally flipping."""
    current = pd4_payoff(bits)
    for idx in range(4):
        flipped = list(bits)
        flipped[idx] ^= 1
        if pd4_payoff(tuple(flipped))[idx] > current[idx]:
            return False
    return True

def find_nash():
    return [bits for bits in itertools.product([0,1], repeat=4)
            if is_nash(bits)]

if __name__ == "__main__":
    start = time.time()
    equilibria = find_nash()
    elapsed = time.time() - start

    print("4-Player PD Nash equilibria:", equilibria)
    print(f"Enumeration time: {elapsed:.6f} s")

    with open("pd4_results.json", "w") as f:
        json.dump({
            "equilibria": equilibria,
            "time_seconds": elapsed
        }, f, indent=2)


4-Player PD Nash equilibria: [(1, 1, 1, 1)]
Enumeration time: 0.000948 s


In [15]:
import json
print(json.load(open("pd4_results.json")))
print(json.load(open("pd3_equilibria.json")))
print(json.load(open("pd2_results.json")))

{'equilibria': [[1, 1, 1, 1]], 'time_seconds': 0.0009479522705078125}
{'equilibria': [[1, 1, 1]], 'time_seconds': 0.00010395050048828125}
{'equilibria': [[1, 1]], 'time_seconds': 3.695487976074219e-05}
